# Import

In [25]:
import sys

sys.path.append(r"D:\Materials\AI\Projects\Lib\torchwires\src")

from torchwires import Repo
from torchwires import BaseCallback
from torchwires import Trainer

In [26]:
import torch
from torch.utils.data import DataLoader, Dataset

import numpy as np

# Create

In [27]:
repo = Repo(
    repo_name="full_test_2",
)

No History cache found: path=full_test_2\exp_1\history.json
No History Features found: path=full_test_2\exp_1\history_tracked.json


# Models

In [28]:
class LinearStack(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = torch.nn.Linear(10, 10)
        self.activation = torch.nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        return x

In [29]:
m1 = LinearStack()

repo.register_model(
    name='m1',
    model=m1
)

Model m1: has been registered
Model m1 weights not found: path=full_test_2\exp_1\last\m1.weights.pth


# Optimizers

In [30]:
repo.register_optimizer(
    name='m1',
    optimizer= torch.optim.Adam(m1.parameters(), lr=0.001)
)

Optimizer m1: has been registered
Optimizer m1 cache not found: path=full_test_2\exp_1\last\m1.optimizer.pth


# Loaders

In [31]:
class MyDataset(Dataset):
    def __init__(self):
        self.err = 0.0

    def __len__(self):
        return 30

    def __getitem__(self, index):
        x = torch.rand((10,))
        y = torch.sin(x) + self.err * torch.randn(10)
        return x, y

In [32]:
train_loader = DataLoader(
        MyDataset(),
        batch_size=2,
)

val_loader = DataLoader(
    MyDataset(),
    batch_size=2,
)

test_loader = DataLoader(
    MyDataset(),
    batch_size=2,
)

# Connections

In [33]:
repo.add_forward(
    model_name='m1',
    inputs=['x'],
    outputs=['y-hat'],
)

ForwardStep added: y-hat = m1(x)


# Losses & Metrics

In [34]:
repo.add_loss(
    loss_name='mae',
    loss_function=lambda s: torch.flatten(s['y-hat'] - s['y']).abs().mean(),
    weight_function= lambda s: 1.0
)

LossStep added: mae


In [35]:
repo.add_metric(
    metric_name='err_inverse',
    metric_function=lambda s: 1 / torch.flatten(s['y-hat'] - s['y']).abs().mean(),
)

MetricStep added: err_inverse


# Callbacks

In [36]:
trainer = Trainer(
    repo=repo,
    device='cpu'
)

In [37]:
class CB_1(BaseCallback):
    def on_epoch_start(
            self
    ):
        print("Epoch started.")

    def on_epoch_end(
            self,
            epoch_state,
    ):
        print(f"Epoch {epoch_state.aggregate_over_batches('epoch', 'train', 'mean')} ended.")
        

trainer.register_callback(CB_1())

In [38]:
trainer.register_checkpoint_callback(
    mode="min",
    monitor="total_loss",
    split="val",
)

In [39]:
trainer.register_checkpoint_callback(
    mode="max",
    monitor="err_inverse",
    split="val",
)

In [40]:
trainer.register_auto_save_callback(
    interval=5,
    checkpoint_name="auto_a",
    concat_with_epoch_no=True
)

In [41]:
trainer.register_auto_save_callback(
    interval=5,
    checkpoint_name="auto_b",
    concat_with_epoch_no=False
)

# Load

In [42]:
repo.load()

Model m1 weights not found: path=full_test_2\exp_1\last\m1.weights.pth
Optimizer m1 cache not found: path=full_test_2\exp_1\last\m1.optimizer.pth
No History cache found: path=full_test_2\exp_1\history.json
No History Features found: path=full_test_2\exp_1\history_tracked.json


# Training

In [43]:
trainer.train(
    n_epochs=20,
    train_loader=train_loader,
    val_loader=val_loader,
    loader_output_keys=['x','y']
)

Training full_test_2: Starting training epoch 0 -> 20 epochs
Epoch started.
Epoch: 1/20                                                                                                                                                                                                                                                                                               
train-mae.raw: 0.30956               | val-mae.raw: 0.29419                 | train-mae.weight: 1.00000            | val-mae.weight: 1.00000             
train-mae.eff: 0.30956               | val-mae.eff: 0.29419                 | train-err_inverse: 3.29578           | val-err_inverse: 3.47128            
train-total_loss: 0.30956            | val-total_loss: 0.29419              | train-m1.lr: 0.00100                 | val-m1.lr: None                     
Epoch 1.0 ended.
Checkpoint taken: val-total_loss improved from inf to 0.294191 | checkpoint name: best_checkpoint_val-total_loss
Model m1 weights saved: path=full

# Access Models

In [44]:
repo.get_all_models_names()

['m1']

In [45]:
repo.get_model_node('m1').get_model()

LinearStack(
  (linear1): Linear(in_features=10, out_features=10, bias=True)
  (activation): ReLU()
)

# Save

In [46]:
repo.save()

Model m1 weights saved: path=full_test_2\exp_1\last\m1.weights.pth
Optimizer m1 weights saved: path=full_test_2\exp_1\last\m1.optimizer.pth


In [47]:
repo.save(
    "final_dat"
)

Model m1 weights saved: path=full_test_2\exp_1\final_dat\m1.weights.pth
Optimizer m1 weights saved: path=full_test_2\exp_1\final_dat\m1.optimizer.pth


# Predict

In [48]:
trainer.predict(
    dataloader=test_loader,
    loader_output_keys=['x','y'],
)

Epoch: 1/1                                                                                                                                                                                                                                                                          
test-mae.raw: 0.21095                | test-mae.weight: 1.00000             | test-mae.eff: 0.21095                | test-err_inverse: 5.03951           
test-total_loss: 0.21095             | test-m1.lr: None                     | 

[{'epoch': 1,
  'batch': 1,
  'loader': 'test',
  'total_loss': tensor(0.1192),
  'x': tensor([[0.9638, 0.9049, 0.4370, 0.7532, 0.4099, 0.5384, 0.6868, 0.5667, 0.5484,
           0.4169],
          [0.8153, 0.5849, 0.5960, 0.3955, 0.7372, 0.8118, 0.5691, 0.9779, 0.5939,
           0.4330]]),
  'y': tensor([[0.8214, 0.7863, 0.4232, 0.6840, 0.3986, 0.5128, 0.6340, 0.5368, 0.5213,
           0.4050],
          [0.7279, 0.5521, 0.5613, 0.3852, 0.6722, 0.7255, 0.5388, 0.8293, 0.5596,
           0.4196]]),
  'y-hat': tensor([[0.7101, 0.7436, 0.5628, 0.4379, 0.5988, 0.5125, 0.5755, 0.4771, 0.5721,
           0.5221],
          [0.5174, 0.5539, 0.6925, 0.5461, 0.4978, 0.5985, 0.4682, 0.5522, 0.6245,
           0.5578]]),
  'mae.raw': tensor(0.1192),
  'mae.weight': 1.0,
  'mae.eff': tensor(0.1192),
  'err_inverse': tensor(8.3925)},
 {'epoch': 1,
  'batch': 2,
  'loader': 'test',
  'total_loss': tensor(0.2961),
  'x': tensor([[0.9236, 0.9458, 0.0395, 0.4319, 0.0907, 0.8358, 0.3223, 0.6248, 0.86